# Level 1 — Risk and Return Foundations

**Audience:** analysts who know basic Python and pandas but are new to the
`AssetManagementToolkit` API.

**Prerequisites:** Python 3.9+, NumPy, pandas, SciPy, and this repository.

**Learning goals**

1. represent periodic simple returns correctly;
2. calculate return, volatility, drawdown, VaR, and risk-adjusted ratios;
3. create one audit-friendly `risk_return_stats` table;
4. interpret frequency and benchmark assumptions.

This notebook distills the reusable ideas from legacy labs 102–106. It uses
synthetic monthly data, so no private dataset or network connection is needed.

## 1. Setup

All examples use **monthly decimal simple returns**. A value of `0.02` means
2%, and therefore `periods_per_year=12`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.analytics.returns import (
    annualized_return,
    total_return,
)
from asset_management_toolkit.analytics.risk import (
    annualized_volatility,
    cornish_fisher_var,
    historical_cvar,
    historical_var,
    max_drawdown,
    sharpe_ratio,
)
from asset_management_toolkit.analytics import risk_return_stats

In [ ]:
dates = pd.period_range("2023-01", periods=24, freq="M").to_timestamp("M")
returns = pd.DataFrame(
    {
        "Balanced": [
            0.020, -0.010, 0.015, 0.008, -0.025, 0.030,
            0.012, 0.006, -0.008, 0.018, 0.011, 0.005,
            0.014, -0.006, 0.021, 0.009, -0.018, 0.026,
            0.010, 0.004, -0.005, 0.016, 0.008, 0.013,
        ],
        "Growth": [
            0.035, -0.028, 0.024, 0.012, -0.050, 0.048,
            0.019, 0.010, -0.017, 0.031, 0.015, 0.008,
            0.023, -0.014, 0.033, 0.016, -0.036, 0.041,
            0.017, 0.007, -0.012, 0.026, 0.013, 0.020,
        ],
    },
    index=dates,
)
returns.head()

## 2. Return and volatility

`total_return` compounds the entire observed path. `annualized_return`
geometrically scales that path to one year. Volatility uses the sample standard
deviation and the square-root-of-time rule.

In [ ]:
pd.DataFrame(
    {
        "total_return": total_return(returns),
        "annualized_return": annualized_return(returns, periods_per_year=12),
        "annualized_volatility": annualized_volatility(
            returns, periods_per_year=12
        ),
        "sharpe_ratio": sharpe_ratio(
            returns, risk_free_rate=0.02, periods_per_year=12
        ),
    }
)

## 3. Drawdown and tail risk

Maximum drawdown is the worst peak-to-trough loss **within the sample path**;
it is not annualized. VaR and CVaR here are positive loss magnitudes. At
`level=0.05`, the functions examine the worst 5% tail.

In [ ]:
pd.DataFrame(
    {
        "max_drawdown": max_drawdown(returns),
        "historical_var_95": historical_var(returns, level=0.05),
        "historical_cvar_95": historical_cvar(returns, level=0.05),
        "cornish_fisher_var_95": cornish_fisher_var(returns, level=0.05),
    }
)

## 4. One summary table

Use the façade when you need a consistent review table rather than calling
each metric separately. The row count makes the observation window explicit.

In [ ]:
stats = risk_return_stats(
    returns,
    risk_free_rate=0.02,
    minimum_acceptable_return=0.00,
    periods_per_year=12,
    var_level=0.05,
)
stats.T

## Exercise

Change the annual risk-free rate from 2% to 4%. Which columns in the summary
should change, and which should remain identical?

In [ ]:
# Try it here.
stats_higher_rf = risk_return_stats(
    returns,
    risk_free_rate=0.04,
    periods_per_year=12,
)

### Answer scaffold

Compare `stats_higher_rf` with `stats`. The Sharpe ratio should change because
it uses the risk-free rate. Return, volatility, drawdown, and standalone tail
risk measures should not.

In [ ]:
changed = stats_higher_rf.ne(stats).any(axis=0)
changed[changed]

## Common pitfalls and next steps

- Match `periods_per_year` to the actual sampling frequency.
- Pass decimal returns, not prices and not percentage points.
- Do not annualize maximum drawdown.
- Tail estimates from 24 observations are illustrative, not decision-grade.

Next: continue to Level 2 to turn expected returns and a covariance matrix into
portfolio weights.